In [ ]:
# Setup: find the course materials and load our helper code. Run this cell first.
import subprocess
import sys
from pathlib import Path

# Colab has a module called google.colab; a laptop does not.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: download the course materials once.
    TASK_DIR = Path("/content/intern-training/tasks/esm2-enzyme-transfer")
    if not TASK_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "esm2-v1",
             "https://github.com/lagillenwater/intern-training", "/content/intern-training"],
            check=True,
        )
    # Install the exact package versions this course was tested with.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 'accelerate==1.16.0', 'transformers==5.23.0', 'umap-learn==0.6.0'],
        check=True,
    )
else:
    # On a laptop: walk up from the current folder until we find the folder
    # that holds both helpers/ and checks.py.
    TASK_DIR = None
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "helpers").is_dir() and (folder / "checks.py").exists():
            TASK_DIR = folder
            break
    if TASK_DIR is None:
        raise RuntimeError("Could not find the course materials (a folder with helpers/ and checks.py).")

# Make our helper code importable, then load it.
sys.path.insert(0, str(TASK_DIR))
from helpers import *
from checks import *

print(f"Setup complete. Materials: {TASK_DIR}")


# 01 Explore the data

**What this step is for:** get to know the data before training any model. How many enzymes of each class does each species have? How long are they, and how many are too long for the model to read in full? Which proteins did we leave out, and why?

**Time:** about 1 hour.

**Reads:** `data/enzymes.tsv` (the cleaned table) and `data/raw/human.tsv` (the human table as downloaded). **Writes:** nothing.

Run the Setup cell at the top first, then work top to bottom. Some cells ask for a prediction or an answer in words. To write in a text cell, double-click it, type under the question, then press Shift+Enter to show it. If a cell below a `TODO` shows an error, first make sure that `TODO` passed its check.

In [ ]:
# Configuration for this run (the same in every notebook).
RUN_NAME = "esm2_8M"                # a label for this run: the small ESM-2 model, with 8 million parameters
WORK_DIR = setup_workdir(RUN_NAME)  # reconnects Google Drive on Colab and returns the run folder
print(WORK_DIR)

In [ ]:
# The standard data tools (notebook 00, Part 4).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Load the data
`df` is the cleaned table we use for the whole project. `raw_human` is the human table exactly as it came from UniProt, the protein database, before we removed anything. Comparing the two at the end of this notebook shows what the cleaning removed.

In [ ]:
df = load_dataset()            # cleaned table: one row per enzyme, all five species
raw_human = load_raw("human")  # human enzymes as downloaded, before cleaning
print(f"df has {len(df):,} rows; raw_human has {len(raw_human):,} rows")
df.head()

## Meet the species
| Name in the table | Organism | What it is |
|---|---|---|
| `human` | *Homo sapiens* | us |
| `mouse` | *Mus musculus* | the house mouse, a mammal |
| `fly` | *Drosophila melanogaster* | the fruit fly, an insect |
| `yeast` | *Saccharomyces cerevisiae* | baker's yeast, a single-celled fungus |
| `ecoli` | *Escherichia coli* | a bacterium |

The model will learn from human enzymes only. Then we test it on held-back human enzymes and on all the enzymes of the other four species.

## The columns
| Column | What it holds |
|---|---|
| `accession` | the protein's ID in UniProt; you can look it up at uniprot.org |
| `species` | which organism the protein comes from |
| `gene`, `protein_name` | the gene and protein names |
| `ec` | the EC number or numbers: a code for the reaction the enzyme speeds up |
| `ec_class` | the first number of the EC number, 1 to 7: the label the model predicts |
| `length` | how many amino acids the protein has |
| `sequence` | the amino acids, one letter each; this is what the model reads |
| `eggnog`, `group_id` | protein family IDs, used to keep close relatives together when we split the human data |
| `split` | `train`, `val`, or `test` for human proteins; every protein from the other species is `test` |

## Enzyme classes
Enzymes are proteins that speed up chemical reactions. An Enzyme Commission (EC) number names the reaction with four numbers, from general to specific. For example, trypsin, a digestive enzyme, is `3.4.21.4`: class 3 (hydrolases), then subclass 4 (it cuts the bonds that hold proteins together), and so on. Some EC numbers stop early, like `3.4.-.-`: the exact reaction is not pinned down, but the class is still known.

The first number, the **class**, is what our model learns to predict. `CLASS_NAMES` from our helpers maps each class number to its name.

In [ ]:
# A plain-language description of each class, as a dict from class number to text.
descriptions = {
    1: "moves electrons between molecules",
    2: "moves a chemical group to another molecule",
    3: "breaks a bond by adding water",
    4: "breaks a bond without water or oxidation",
    5: "rearranges the atoms within one molecule",
    6: "joins two molecules, powered by ATP",
    7: "carries ions or molecules across a membrane",
}
# Build a table from two dicts that share the same keys: each dict becomes a column, the keys become the rows.
pd.DataFrame({"name": CLASS_NAMES, "what it does": descriptions})

## Classes per species
Next you count the enzymes of each class in each species. A *cross-tabulation* is a table of counts for every pair of values from two columns: here, every (species, class) pair.

**Prediction (write this before you run the next cell):** Which enzyme class do you think is most common in human? Which is rarest? Why?

In [ ]:
# TODO: Make a table named counts with one row per species and one column per EC class,
#       where each cell holds how many enzymes of that species are in that class.
# Hint: use pd.crosstab(df["species"], df["ec_class"])
counts = None  # replace None with your code

In [ ]:
check_class_counts(counts, df)

In [ ]:
# Show your table with the species in the order we use everywhere, human first.
# .loc[...] picks rows by their labels, in the order you list them.
counts.loc[SPECIES_ORDER]

In [ ]:
# The species have different numbers of enzymes, so raw counts are hard to compare.
# normalize="index" divides each row by its total, so each species' row adds up to 1.
shares = pd.crosstab(df["species"], df["ec_class"], normalize="index")
shares.loc[SPECIES_ORDER].round(2)  # round to 2 decimal places so the table is easier to read

**Compare:** Was your prediction right? What surprised you?

## The score to beat
*Accuracy* is the fraction of guesses a model gets right. Imagine a lazy "model" that ignores the sequence and always guesses the class that is most common in the training data. What would its accuracy be? Any real model has to do better than this. We measure it on the human training proteins (`split` is `"train"`), the ones the model will learn from.

In [ ]:
# TODO: Keep only the human training proteins in human_train. Then compute majority_accuracy: the fraction
#       of human_train that is in its most common class (a number between 0 and 1).
# Hint: filter df with (df["species"] == "human") & (df["split"] == "train"), then use
#       human_train["ec_class"].value_counts(normalize=True).max()   (normalize=True gives fractions instead of counts)
human_train = None  # replace None with the human training rows
majority_accuracy = None  # replace None with your code

In [ ]:
check_majority_accuracy(majority_accuracy, df)

In [ ]:
# :.1% shows a fraction as a percentage with one decimal place (notebook 00, Part 5).
print(f"Always guessing the most common class is right {majority_accuracy:.1%} of the time on human training proteins.")

## Protein lengths
ESM-2, the model you will fine-tune, reads at most `MAX_RESIDUES` (1,022) amino acids of a protein. For a longer protein it reads only the first 1,022 and never sees the rest. This is called *truncation*. How many of our proteins are that long?

A *histogram* groups values into bins (here, ranges of length) and draws a bar for how many values fall in each bin. A few proteins are many thousands of amino acids long, so the plot stops at 4,000 to keep the rest readable.

In [ ]:
# TODO: Draw a histogram of protein lengths from 0 to 4000, with a vertical line at MAX_RESIDUES. Label both axes.
# Hint: use plt.hist(df["length"], bins=80, range=(0, 4000)) and plt.axvline(MAX_RESIDUES, color="black", linestyle="--"),
#       then plt.xlabel, plt.ylabel, and plt.show() as in notebook 00.
# YOUR CODE HERE

In [ ]:
# TODO: Compute fraction_truncated: the fraction of all proteins (every species) that are longer than MAX_RESIDUES.
# Hint: make a mask with df["length"] > MAX_RESIDUES, then call .mean() on it (True counts as 1 and False as 0).
fraction_truncated = None  # replace None with your code

In [ ]:
check_fraction_truncated(fraction_truncated, df)

In [ ]:
# Say the result in words.
print(f"{fraction_truncated:.1%} of the enzymes are longer than the model can read in full.")

## Enzymes with more than one class
Some enzymes speed up more than one reaction, so UniProt lists several EC numbers, separated by semicolons. Sometimes those numbers fall in different classes. Our model gives each protein exactly one class, so the cleaning step dropped every protein whose EC numbers span more than one class. Here you count how many human proteins that removed.

Our helper `ec_classes` turns an EC string into the **set** of classes it mentions. A Python set is like a Java `HashSet`: it is written with curly braces and never holds the same item twice.

In [ ]:
print(ec_classes("3.4.21.4"))                # one EC number: one class
print(ec_classes("1.1.1.1; 1.1.1.2"))        # two EC numbers in the same class: still one class
print(ec_classes("2.7.7.7; 3.1.11.1"))       # two EC numbers in different classes: two classes
print(len(ec_classes("2.7.7.7; 3.1.11.1")))  # len() of a set is how many items it holds

In [ ]:
# .map(f) calls the function f on every value of a column and returns the results as a new Series,
# with no loop to write (like Java's stream().map(f)). Pass the function's name without parentheses.
# Here: the set of classes for the first five rows of raw_human.
raw_human["ec"].head().map(ec_classes)

In [ ]:
# TODO: Count the proteins in raw_human whose EC numbers span more than one class. Store the count in n_multiclass.
# Hint: raw_human["ec"].map(ec_classes) gives each protein's set of classes, and .map(len) on that gives each set's size;
#       then (sizes > 1).sum() counts the sizes bigger than 1.
n_multiclass = None  # replace None with your code

In [ ]:
check_multiclass_count(n_multiclass, raw_human)

In [ ]:
# Does it add up? The raw human rows minus the dropped ones should equal the human rows in the cleaned table.
n_human_clean = (df["species"] == "human").sum()
print(f"raw: {len(raw_human):,}   dropped: {n_multiclass:,}   kept in df: {n_human_clean:,}")
print("Adds up:", len(raw_human) - n_multiclass == n_human_clean)

## In your own words
**Question 1.** The classes are not equally common. Look back at your counts table and your `majority_accuracy`. Why does this imbalance change how we should score a model? What could a model with high accuracy still get wrong?

**Your answer:**

**Question 2.** Why did we drop the enzymes whose EC numbers span more than one class? Think about what the model is asked to predict for each protein.

**Your answer:**

**Prediction (write this now; you will compare it with the real scores in notebook 04):** The model learns enzyme classes from human proteins only. Which test species (mouse, fly, yeast, or ecoli) do you think it will do worst on, and why?

## Done
Next: notebook 02, where you build a first model, the baseline, to compare fine-tuning against.